In [1]:
# smiles_list
water_smiles = "O"
ethanol_smiles = "CCO"
Aspirin = "CC(=O)OC1=CC=CC=C1C(=O)O"
smiles = [water_smiles, ethanol_smiles, Aspirin]
for smile in smiles:
    print(smile)
    name_length = len(smile)
    print(f"Molecule {smile} has {name_length} characters")


O
Molecule O has 1 characters
CCO
Molecule CCO has 3 characters
CC(=O)OC1=CC=CC=C1C(=O)O
Molecule CC(=O)OC1=CC=CC=C1C(=O)O has 24 characters


In [2]:
chemical_library = {
    "Water": "O",
    "Ethanol": "CCO",
    "Dopamine": "C1(O)=C(O)C=CC(CCN)=C1",
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O"
}
for molecule_name, smile in chemical_library.items():
    smile_length = len(smile)
    if smile_length < 5:
        classification = "Tiny"
    elif smile_length <= 15:
        classification = "Medium"
    else:
        classification = "Large"
    print(f"Molecule {molecule_name} ({smile}) is classified as {classification}")

Molecule Water (O) is classified as Tiny
Molecule Ethanol (CCO) is classified as Tiny
Molecule Dopamine (C1(O)=C(O)C=CC(CCN)=C1) is classified as Large
Molecule Aspirin (CC(=O)OC1=CC=CC=C1C(=O)O) is classified as Large


In [3]:
from rdkit import Chem
chemical_library = {
    "Water": "O",
    "Ethanol": "CCO",
    "Dopamine": "C1(O)=C(O)C=C(CCN)C=C1",
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O"
}
for name, smiles in chemical_library.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        heavy_atom_count = mol.GetNumHeavyAtoms()
        raw_string_length = len(smiles)
        if heavy_atom_count < 5:
            classification = "Tiny"
        elif heavy_atom_count <= 12:
            classification = "Medium"
        else:
            classification = "Large"
        print(f"{name}:")
        print(f" - Raw Text Length: {raw_string_length}")
        print(f" - True Heavy Atoms: {heavy_atom_count}")
        print(f" - Classfication: {classification}")
        print("-" * 40)
    else:
        print(f"Error: Could not parse SMILES for {name}")

Water:
 - Raw Text Length: 1
 - True Heavy Atoms: 1
 - Classfication: Tiny
----------------------------------------
Ethanol:
 - Raw Text Length: 3
 - True Heavy Atoms: 3
 - Classfication: Tiny
----------------------------------------
Dopamine:
 - Raw Text Length: 22
 - True Heavy Atoms: 11
 - Classfication: Medium
----------------------------------------
Aspirin:
 - Raw Text Length: 24
 - True Heavy Atoms: 13
 - Classfication: Large
----------------------------------------


In [15]:
from rdkit import Chem
file_path = "compounds.smi"
print(f"Reading chemistry_library from: {file_path}\n")
with open(file_path, "r") as file:
    for line in file:
        line = line.strip()
        if not line:
            continue
        parts = line.split("\t")
        if len(parts) == 2:
            smiles, name = parts[0], parts[1]
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                atoms = mol.GetNumHeavyAtoms()
                if atoms < 5:
                    classification = "Tiny"
                elif atoms <= 12:
                    classification = "Medium"
                else:
                    classification = "Large"
                print(f"{name:<10} | SMILES: {smiles:<30} | Heavy Atoms: {atoms:<2} | Class: {classification}")
            else:
                print(f"Error parsing SMILES for {name}")

Reading chemistry_library from: compounds.smi

Water      | SMILES: O                              | Heavy Atoms: 1  | Class: Tiny
Ethanol    | SMILES: CCO                            | Heavy Atoms: 3  | Class: Tiny
Dopamine   | SMILES: C1(O)=C(O)C=C(CCN)C=C1         | Heavy Atoms: 11 | Class: Medium
Aspirin    | SMILES: CC(=O)OC1=CC=CC=C1C(=O)O       | Heavy Atoms: 13 | Class: Large
Caffeine   | SMILES: CN1C=NC2=C1C(=O)N(C(=O)N2C)C   | Heavy Atoms: 14 | Class: Large
Ibuprofen  | SMILES: CC(C)CC1=CC=C(C=C1)C(C)C(=O)O  | Heavy Atoms: 15 | Class: Large


In [21]:
from rdkit import Chem
from rdkit.Chem import Descriptors
file_path = "compounds.smi"
print(f"{'Compound':<12} | {'MW':<7} | {'LogP':<6} | {'HBD':<4} | {'HBA':<4} | {'Lipinski Status'}")
print("-" * 65)

with open(file_path, "r") as file:
    for line in file:
        line = line.strip()
        if not line:
            continue

        parts = line.split("\t")
        if len(parts) == 2:
            smiles, name = parts[0], parts[1]
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                mw = Descriptors.MolWt(mol)
                logp = Descriptors.MolLogP(mol)
                hbd = Descriptors.NumHDonors(mol)
                hba = Descriptors.NumHAcceptors(mol)

                violations = 0
                if mw >= 500: violations += 1
                if logp >= 5: violations += 1
                if hbd > 5: violations += 1
                if hba > 10: violations += 1

                status = "PASS" if violations <= 1 else f"FAIL ({violations} violations)"

                print(f"{name:<12} | {mw:<7.2f} | {logp:<6.2f} | {hbd:<4} | {hba:<4} | {status}")
            else:
                print(f"ERROR parsing SMILES for {name}")
                

            

Compound     | MW      | LogP   | HBD  | HBA  | Lipinski Status
-----------------------------------------------------------------
Water        | 18.02   | -0.82  | 0    | 0    | PASS
Ethanol      | 46.07   | -0.00  | 1    | 1    | PASS
Dopamine     | 153.18  | 0.60   | 3    | 3    | PASS
Aspirin      | 180.16  | 1.31   | 1    | 3    | PASS
Caffeine     | 194.19  | -1.03  | 0    | 6    | PASS
Ibuprofen    | 206.28  | 3.07   | 1    | 1    | PASS


In [22]:
from rdkit import Chem
chemical_database = {
    "Water": "O",
    "Ethanol": "CCO",
    "Corrupted_Compound": "C1CC1CC1(CC",
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O"
}
for name, smiles in chemical_database.items():
    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        print(f"[{name}] parsed successfully!")
    else:
        print(f"WARNING: [{name}] has a corrupted SMILES strings!")
        

[Water] parsed successfully!
[Ethanol] parsed successfully!
[Aspirin] parsed successfully!


In [23]:
from rdkit import Chem
from rdkit.Chem import Descriptors

chemical_database = {
    "Water": "O",
    "Ethanol": "CCO",
    "Corrupted_Compound": "C1CC1CC1(CC",
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O"
}
for name, smiles in chemical_database.items():
    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        mw = Descriptors.ExactMolWt(mol)
        print(f"Molecule [{name}] has a Molecular Weight of {mw:.2f} g/mol")
    else:
        print(f"Skipping [{name}] because it is invalid")

Molecule [Water] has a Molecular Weight of 18.01 g/mol
Molecule [Ethanol] has a Molecular Weight of 46.04 g/mol
Skipping [Corrupted_Compound] because it is invalid
Molecule [Aspirin] has a Molecular Weight of 180.04 g/mol


In [25]:
import pandas as pd
clean_rows = []
for name, smiles in chemical_database.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        mw = Descriptors.ExactMolWt(mol)
        clean_rows.append({
            "Name|": name,
            "SMILES|": smiles,
            "Molecular_Weight|": round(mw, 2)
        })
df = pd.DataFrame(clean_rows)
df.to_csv("clean_molecules.csv", index = False)

print("SUCCESS: 'clean_molecules.csv' generated!")
print(df)


SUCCESS: 'clean_molecules.csv' generated!
     Name|                   SMILES|  Molecular_Weight|
0    Water                         O              18.01
1  Ethanol                       CCO              46.04
2  Aspirin  CC(=O)OC1=CC=CC=C1C(=O)O             180.04
